In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

# Memory buffer for on-policy rollouts
class Memory:
    def __init__(self):
        self.accel_x = []
        self.accel_y = []
        self.accel_z = []
        self.gyro_x = []
        self.gyro_y = []
        self.gyro_z = []

    def store(self, accel, gyro):
        self.accel_x.append(accel[0])
        self.accel_y.append(accel[1])
        self.accel_z.append(accel[2])
        self.gyro_x.append(gyro[0])
        self.gyro_y.append(gyro[1])
        self.gyro_z.append(gyro[2])

    def clear(self):
        self.__init__()

In [ ]:
class Actor(Model):
    def __init__(self, act_dim):
        super().__init__()
        
        self.d1 = layers.Dense(64, activation='tanh')
        self.d2 = layers.Dense(64, activation='tanh')
        self.mu = layers.Dense(act_dim)
        
        self.log_std = self.add_weight(
            name='log_std',
            shape=(act_dim,),
            initializer=tf.constant_initializer(-0.5),
            trainable=True
        )
        
    def call(self, input):
        x = self.d1(input)
        x = self.d2(x)
        mu = self.mu(x)
        
        batch_size = tf.shape(input)[0]
        log_std = tf.broadcast_to(self.log_std[None, :], (batch_size, self.log_std.shape[0]))
        return mu, log_std

In [ ]:
import tensorflow_probability as tfp

class PPOAgent:
    def __init__(self, act_dim, 
                 clip_ratio=0.2, gamma=0.99, lam=0.95,
                 pi_lr=3e-4, vf_lr=1e-3, train_pi_iters=80, train_v_iters=80, action_scale=0.3, smooth_coef=0.5):
        self.act_dim = act_dim
        self.clip_ratio = clip_ratio
        self.gamma = gamma
        self.lam = lam
        self.train_pi_iters = train_pi_iters
        self.train_v_iters = train_v_iters
        self.action_scale = action_scale
        self.smooth_coef = smooth_coef
        self.prev_action = np.zeros(act_dim, dtype=np.float32)

        # Build actor and critic
        self.actor = Actor(self.act_dim)
        self.critic = self._build_critic()
        self.actor_optimizer = tf.keras.optimizers.Adam(learning_rate=pi_lr)
        self.critic_optimizer = tf.keras.optimizers.Adam(learning_rate=vf_lr)
        self.memory = Memory()

    def _build_critic(self):
        inp = layers.Input(shape=(self.obs_dim,))
        x = layers.Dense(64, activation='tanh')(inp)
        x = layers.Dense(64, activation='tanh')(x)
        v = layers.Dense(1)(x)
        return Model(inputs=inp, outputs=v)

    def act(self, obs):
        tfd = tfp.distributions
        obs = obs.reshape(1, -1).astype(np.float32)
        mu, log_std = self.actor(obs)
        std = tf.exp(log_std)
        pi_dist = tfd.Normal(loc=mu, scale=std)
        pi = pi_dist.sample()
        logp = tf.reduce_sum(pi_dist.log_prob(pi), axis=-1)
        value = self.critic(obs)
        
        raw_action = pi.numpy()[0]

        # 4) 액션 스케일링 (크기를 줄인다)
        scaled = raw_action * self.action_scale

        # 5) 지수 이동 평균 스무딩
        smoothed = (self.smooth_coef * self.prev_action
                    + (1 - self.smooth_coef) * scaled)
        self.prev_action = smoothed

        # 6) 환경 허용 범위로 클리핑
        action = np.clip(smoothed, -1, 1)

        return action, logp.numpy()[0], value.numpy()[0,0]

    def store(self, obs, action, logp, value, reward, done):
        self.memory.store(obs, action, logp, value, reward, done)

    def compute_gae(self, last_value):
        rewards = np.array(self.memory.rewards + [0], dtype=np.float32)
        values = np.array(self.memory.values + [last_value], dtype=np.float32)
        dones = np.array(self.memory.dones + [0], dtype=np.float32)
        T = len(self.memory.rewards)
        adv = np.zeros(T, dtype=np.float32)
        last_gae = 0
        for t in reversed(range(T)):
            delta = rewards[t] + self.gamma * values[t+1] * (1-dones[t]) - values[t]
            adv[t] = last_gae = delta + self.gamma * self.lam * (1-dones[t]) * last_gae
        returns = adv + values[:-1]
        return adv, returns

    def update(self):
        obs = np.array(self.memory.obs, dtype=np.float32)
        actions = np.array(self.memory.actions, dtype=np.float32)
        old_logps = np.array(self.memory.logps, dtype=np.float32)
        values = np.array(self.memory.values, dtype=np.float32)
        last_value = values[-1]
        adv, returns = self.compute_gae(last_value)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        # Policy update
        for _ in range(self.train_pi_iters):
            with tf.GradientTape() as tape:
                mu, log_std = self.actor(obs)
                std = tf.exp(log_std)
                new_logp = -0.5 * tf.reduce_sum(((actions - mu)/std)**2 + 2*log_std + np.log(2*np.pi), axis=1)
                ratio = tf.exp(new_logp - old_logps)
                clipped_ratio = tf.clip_by_value(ratio, 1 - self.clip_ratio, 1 + self.clip_ratio)
                pi_loss = -tf.reduce_mean(tf.minimum(ratio * adv, clipped_ratio * adv))
            grads = tape.gradient(pi_loss, self.actor.trainable_variables)
            self.actor_optimizer.apply_gradients(zip(grads, self.actor.trainable_variables))

        # Value update
        for _ in range(self.train_v_iters):
            with tf.GradientTape() as tape:
                v = self.critic(obs)[:,0]
                v_loss = tf.reduce_mean((returns - v)**2)
            grads = tape.gradient(v_loss, self.critic.trainable_variables)
            self.critic_optimizer.apply_gradients(zip(grads, self.critic.trainable_variables))

        self.memory.clear()

In [ ]:
import serial
import time

arduino = serial.Serial("COM7", 9600)
time.sleep(2)

while True:
    if arduino.in_waiting > 0:
        data = arduino.readline().decode('utf-8').strip()
        
        parts = data.split()  # 공백 기준으로 나누기
        accel_x = int(parts[1][2:])  # 1G = 13000으로 근사
        accel_y = int(parts[2][2:])  
        accel_z = int(parts[3][2:])  

        gyro_x = int(parts[6][2:])   # 1도/초 = 130으로 근사
        gyro_y = int(parts[7][2:])   
        gyro_z = int(parts[8][2:])
        
        


센서값 :  MPU6050 초기화 중...
센서값 :  MPU6050 연결 성공!
센서값 :  Accel: X=-2840 Y=-15772 Z=4892 || Gyro: X=-1047 Y=-101 Z=-116
센서값 :  Accel: X=-2912 Y=-15620 Z=4544 || Gyro: X=-817 Y=-70 Z=-96
센서값 :  Accel: X=-2808 Y=-15708 Z=4664 || Gyro: X=-822 Y=-60 Z=-115
센서값 :  Accel: X=-2872 Y=-15664 Z=4488 || Gyro: X=-946 Y=-49 Z=-70
센서값 :  Accel: X=7864 Y=-8148 Z=-7516 || Gyro: X=-475 Y=4453 Z=7475
센서값 :  Accel: X=-3128 Y=-6192 Z=13900 || Gyro: X=-1354 Y=619 Z=4105
센서값 :  Accel: X=-1272 Y=-14012 Z=9848 || Gyro: X=-643 Y=251 Z=-60
센서값 :  Accel: X=-1616 Y=-13964 Z=9688 || Gyro: X=-908 Y=-155 Z=-97
센서값 :  Accel: X=-924 Y=-13480 Z=5956 || Gyro: X=-6236 Y=6364 Z=-1264
센서값 :  Accel: X=5716 Y=-16412 Z=4612 || Gyro: X=-5567 Y=-8086 Z=-25450
센서값 :  Accel: X=18568 Y=-2644 Z=-1512 || Gyro: X=-1954 Y=-209 Z=813
센서값 :  Accel: X=12500 Y=-5012 Z=2936 || Gyro: X=-32364 Y=-8699 Z=-3748
센서값 :  Accel: X=12368 Y=-3508 Z=9068 || Gyro: X=26730 Y=27417 Z=-3089
센서값 :  Accel: X=2384 Y=11736 Z=13076 || Gyro: X=14778 Y=32767 Z=-2149

SerialException: ClearCommError failed (PermissionError(13, '장치가 명령을 인식하지 않습니다.', None, 22))